# Gym Churn: Data Preparation and EDA

This notebook loads and checks the Model Fitness customer data, then explores how customers who churned differ from those who stayed.

In [1]:
# Libraries for data handling and visualization
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)  # show all columns in outputs

In [2]:
# Load the customer dataset
gym_data = pd.read_csv('../data/gym_churn_us.csv')

In [3]:
# Dataset size and first rows
print(gym_data.shape)
gym_data.head()

(4000, 14)


,gender,Near_Location,Partner,Promo_friends,Phone,Contract_period,Group_visits,Age,Avg_additional_charges_total,Month_to_end_contract,Lifetime,Avg_class_frequency_total,Avg_class_frequency_current_month,Churn
0,1,1,1,1,0,6,1,29,14.227470,5.0,3,0.020398,0.000000,0
1,0,1,0,0,1,12,1,31,113.202938,12.0,7,1.922936,1.910244,0
2,0,1,1,0,1,1,0,28,129.448479,1.0,2,1.859098,1.736502,0
3,0,1,1,1,1,12,1,33,62.669863,12.0,2,3.205633,3.357215,0
4,1,1,1,1,1,1,0,26,198.362265,1.0,3,1.113884,1.120078,0


In [4]:
# Data types and non-null counts per column
gym_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 4000 entries, 0 to 3999
Data columns (total 14 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   gender                             4000 non-null   int64  
 1   Near_Location                      4000 non-null   int64  
 2   Partner                            4000 non-null   int64  
 3   Promo_friends                      4000 non-null   int64  
 4   Phone                              4000 non-null   int64  
 5   Contract_period                    4000 non-null   int64  
 6   Group_visits                       4000 non-null   int64  
 7   Age                                4000 non-null   int64  
 8   Avg_additional_charges_total       4000 non-null   float64
 9   Month_to_end_contract              4000 non-null   float64
 10  Lifetime                           4000 non-null   int64  
 11  Avg_class_frequency_total          4000 non-null   float64
 12  Avg

In [5]:
# Check data quality: missing values and duplicated rows
print("Missing values:", gym_data.isna().sum().sum())
print("Duplicated rows:", gym_data.duplicated().sum())

Missing values: 0
Duplicated rows: 0


In [6]:
# Standardize column names to lowercase for consistency
gym_data.columns = gym_data.columns.str.lower()
gym_data.columns

Index(['gender', 'near_location', 'partner', 'promo_friends', 'phone',
       'contract_period', 'group_visits', 'age',
       'avg_additional_charges_total', 'month_to_end_contract', 'lifetime',
       'avg_class_frequency_total', 'avg_class_frequency_current_month',
       'churn'],
      dtype='str')

In [7]:
# Check the values in the 'month_to_end_contract' column
gym_data['month_to_end_contract'].sort_values().unique()

array([ 1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10., 11., 12.])

In [8]:
# All values are whole months, so convert to int
gym_data['month_to_end_contract'] = gym_data['month_to_end_contract'].astype(
    int)
gym_data['month_to_end_contract'].dtype

dtype('int64')

In [9]:
# Confirm the available contract lengths (in months)
gym_data['contract_period'].value_counts()

contract_period
1     2207
12     960
6      833
Name: count, dtype: int64

In [10]:
# Check that binary features only contain 0 and 1
binary_cols = ['gender', 'near_location', 'partner', 'promo_friends',
               'phone', 'group_visits', 'churn']

for col in binary_cols:
    print(col, gym_data[col].sort_values().unique())

gender [0 1]
near_location [0 1]
partner [0 1]
promo_friends [0 1]
phone [0 1]
group_visits [0 1]
churn [0 1]


### Observations
- The dataset has 4,000 customers and 14 numeric features, with no missing values or duplicates.
- Column names were converted to lowercase, and `month_to_end_contract` was converted to int since it only holds whole months. 
- Binary features (gender, near_location, partner, promo_friends, phone, group_visits, churn) only contain 0 and 1. They are kept as integers rather than booleans: their mean directly gives the share of customers with each characteristic, and the models used later require numeric inputs.